# NumCompute quickstart

End-to-end walk through the package: CSV in, preprocessing, statistics, sort/rank, metrics, gradients, pipeline and a benchmark.

In [ ]:
import numpy as np
import sys
sys.path.insert(0, "..")  # Find numcompute sample csv in the parent folder.

from numcompute import io
from numcompute import preprocessing as pp
from numcompute import stats
from numcompute.utils import iter_batches

sample_path = "data/sample.csv"

np.set_printoptions(precision=3, suppress=True)

## 1. Read CSV (`io.py`)

`data/sample.csv` has 1,200 rows (age, income, city) with missing cells written in several ways: `""`, `NA`, `NaN`, `NULL` and `N/A`. All of them are in `io.DEFAULT_MISSING`, so `load_csv` treats them as missing by default: numeric columns become NaN and the text column becomes `''`. `on_missing='skip'` drops incomplete rows instead, and `iter_csv_chunks` streams the same file in fixed-size chunks.

In [ ]:
X_num, num_names = io.load_csv(sample_path, usecols=['age', 'income'], dtype=float, return_header=True)
city = io.load_csv(sample_path, usecols='city')
X = io.load_csv(sample_path)
print(num_names, X_num.shape, 'missing per column:', np.isnan(X_num).sum(axis=0), '| missing cities:', (city == '').sum())
print(X_num[:5])
print(city[:5].ravel())

print('complete rows only:', io.load_csv(sample_path, usecols=num_names, dtype=float, on_missing='skip').shape)

# same file streamed 500 rows at a time; stacking the chunks gives the same array
chunks = list(io.iter_csv_chunks(sample_path, chunk_size=500, usecols=num_names, dtype=float))
print([c.shape for c in chunks], np.array_equal(np.vstack(chunks), X_num, equal_nan=True))

## 2. Preprocess (`preprocessing.py`)

Impute missing numbers, then scale. Fit on train, apply to new rows with the same statistics.

In [ ]:
imp = pp.SimpleImputer(strategy='median')
X_full = imp.fit_transform(X_num)
print('fill values:', imp.statistics_)

scaler = pp.StandardScaler().fit(X_full)
Z = scaler.transform(X_full)
print('mean after scaling:', Z.mean(axis=0), ' std:', Z.std(axis=0))

print(pp.MinMaxScaler((0, 1)).fit_transform(X_full)[:3])

In [ ]:
enc = pp.OneHotEncoder(handle_unknown='ignore').fit(city)
print('categories:', enc.categories_[0])
print(enc.transform(city[:5]))       # blank city -> all zeros
print(enc.transform([['Darwin']]))   # unseen city, ignored

In [ ]:
# scaler fitted on 2 features rejects a 1-feature matrix with a clear message
try:
    scaler.transform(X_full[:, :1])
except ValueError as e:
    print('ValueError:', e)

## 3. Descriptive statistics (`stats.py`)

`axis=0` summarises each column. `skip_nan=True` ignores NaN; by default NaN propagates like NumPy.

In [ ]:
print('mean   ', stats.mean(X_num, axis=0))
print('mean   ', stats.mean(X_num, axis=0, skip_nan=True), '(skip NaN)')
print('median ', stats.median(X_num, axis=0, skip_nan=True))
print('std    ', stats.std(X_num, axis=0, ddof=1, skip_nan=True))
print('min/max', stats.min(X_num, axis=0, skip_nan=True), stats.max(X_num, axis=0, skip_nan=True))

In [ ]:
print(stats.quantile(X_num, [0.25, 0.5, 0.75], axis=0, skip_nan=True))   # shape (3, n_features)
counts, edges = stats.histogram(X_num[:, 1], bins=4)
print('income histogram:', counts, edges)

In [ ]:
# streaming stats: same numbers as the batch version, O(d) memory
rs = stats.RunningStats(skip_nan=True)
for batch in iter_batches(X_num, 3):
    rs.update(batch)
print('rows seen:', rs.n)
print('running mean:', rs.mean)
print('running std :', rs.std(ddof=1))

## 4. Sort, search, top-k and rank (`sort_search.py`, `rank.py`)

TODO (Faris): top-k on income, quickselect, binary search, rank with ties, percentile.

In [ ]:
# TODO (Faris)

## 5. Metrics (`metrics.py`)

The toolkit has no models, so this section scores a **simulated** income estimator: the true income plus Gaussian noise (sd $20,000, fixed seed), on the rows of `data/sample.csv` that have an income. The same estimate is evaluated as a regression, as a binary classifier (high earner: income above the median) and as a 3-class classifier (income bands).

### 5.1 Regression: mean squared error

The estimate is compared with a baseline that always predicts the mean income. RMSE (the square root of `mse`) is in dollars.

In [ ]:
from numcompute import metrics

income = X_num[:, 1][~np.isnan(X_num[:, 1])]   # rows with a known income
rng = np.random.default_rng(0)
income_hat = income + rng.normal(0, 20_000, size=income.size)   # simulated estimate

baseline = np.full_like(income, income.mean())
print('rows with income:', income.size)
print(f"RMSE estimate:      {np.sqrt(metrics.mse(income, income_hat)):>9,.0f}")
print(f"RMSE mean baseline: {np.sqrt(metrics.mse(income, baseline)):>9,.0f}")

### 5.2 Binary classification: high earners

The labels are text (`'high'` / `'low'`), so the positive class is given as `pos_label='high'`. In the confusion matrix, rows are the true class and columns the predicted one.

In [ ]:
cut = np.median(income)
y_true = np.where(income > cut, 'high', 'low')
y_pred = np.where(income_hat > cut, 'high', 'low')

print(metrics.confusion_matrix(y_true, y_pred, labels=['low', 'high']))   # rows/columns: low, high
print('accuracy ', round(metrics.accuracy(y_true, y_pred), 3))
print('precision', round(metrics.precision(y_true, y_pred, pos_label='high'), 3))
print('recall   ', round(metrics.recall(y_true, y_pred, pos_label='high'), 3))
print('f1       ', round(metrics.f1(y_true, y_pred, pos_label='high'), 3))

### 5.3 ROC curve and AUC

The estimate itself is the score: each threshold flags every estimate at or above it as `'high'`. The AUC is the chance that a random high earner gets a higher estimate than a random low earner (0.5 = guessing, 1.0 = perfect ranking).

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_true, income_hat, pos_label='high')

print('points on the curve:', thresholds.size)
for i in np.linspace(0, thresholds.size - 1, 6).astype(int):   # a few points, highest threshold first
    print(f"threshold {thresholds[i]:>10,.0f}   fpr {fpr[i]:.2f}   tpr {tpr[i]:.2f}")
print('AUC:', round(metrics.auc(fpr, tpr), 3))

### 5.4 Three classes: income bands

The bands split income at its tertiles. `average=None` returns one score per class in **sorted** label order (`high`, `low`, `mid`), so the classes are printed next to the scores. The middle band is the hardest: it can be missed on both sides.

In [ ]:
edges = np.quantile(income, [1 / 3, 2 / 3])
bands = np.array(['low', 'mid', 'high'])
band_true = bands[np.searchsorted(edges, income)]
band_pred = bands[np.searchsorted(edges, income_hat)]

print(metrics.confusion_matrix(band_true, band_pred, labels=bands))   # rows/columns: low, mid, high
print('classes:     ', np.unique(band_true))
print('f1 per class:', metrics.f1(band_true, band_pred, average=None))
for avg in ('macro', 'micro', 'weighted'):
    print(f"f1 {avg:<9}", round(metrics.f1(band_true, band_pred, average=avg), 3))

### 5.5 Informative errors

Labels read with `io.load_csv` are text by default. Comparing text labels with integer predictions raises a `TypeError` instead of silently scoring 0.

In [ ]:
band_codes = np.searchsorted(edges, income_hat)   # the same predictions, as integers 0/1/2
try:
    metrics.accuracy(band_true, band_codes)
except TypeError as err:
    print('TypeError:', err)

## 6. Pipeline (`pipeline.py`)

A Pipeline runs named steps in order. Here we use the same CSV data loaded as `X`. Each step receives the output of the previous step.

### 6.1 Fill and standardize numerical columns

Use the first 900 rows to fit the pipeline. Fill missing age/income values with 0, then standardize each column. Zero filling is used only as a simple example.

In [ ]:
from numcompute.pipeline import Pipeline
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.preprocessing import SimpleImputer as Imputer

# numeric_input = X[:, :2].astype(float)
numeric_input = io.load_csv(sample_path, usecols=["age", "income"], dtype=float)
X_train = numeric_input[:900]
X_test = numeric_input[900:]

numeric_pipe = Pipeline([
    ("impute", Imputer(strategy="constant",fill_value=0)),
    ("scale", StandardScaler()),
])

X_train_ready = numeric_pipe.fit_transform(X_train)
print("Training shape:", X_train_ready.shape)
print(X_train_ready[:5])
print("Training column means:", np.round(X_train_ready.mean(axis=0), 6))

### 6.2 Reuse the fitted pipeline

Call `transform` on the remaining 300 rows. This reuses the parameters learned from the first 900 rows and does not fit again.

In [ ]:
X_test_ready = numeric_pipe.transform(X_test)
print("Test shape:", X_test_ready.shape)
print(X_test_ready[:5])

### 6.3 Fill and encode cities

For the categorical column, replace NaN with `Unknown`, then apply one-hot encoding.

In [ ]:
city_input = X[:, 2:3]
blank_city = city_input[:, 0] == ""

city_imputer = Imputer(strategy="constant", fill_value="Unknown")
city_encoder = OneHotEncoder()
city_pipe = Pipeline([
    ("impute", city_imputer),
    ("encode", city_encoder),
])

cities_ready = city_pipe.fit_transform(city_input)
filled_cities = city_imputer.transform(city_input)

print("Blank cities before imputation:", blank_city.sum())
print("Filled values:", filled_cities[blank_city][:5].ravel())
print("Categories:", city_encoder.categories_[0])
print("Encoded city shape:", cities_ready.shape)
print("Encoded rows for originally blank cities:")
print(cities_ready[blank_city][:5])


### 6.4 Fit and predict with an estimator

Transformers provide `fit(X)` and `transform(X)`; estimators provide
`fit(X, y)` and `predict(X)`. Pipeline calls these methods in order.

`GPT10_5_Astra` is a small demonstration estimator: it remembers the most
common training city and returns it for every prediction. It does not learn
an age/income relationship. Python class names cannot contain a dot, so the
display name GPT10.5_Astra uses an underscore in code.

Use age and income as features and city as the target. Exclude blank training
labels. This demonstration does not use the CSV's simulated `city_predict`.

In [ ]:
class GPT10_5_Astra:
    """Always predict the most common training city."""

    def fit(self, X, y):
        print("model.fit works!")
        labels, counts = np.unique(y, return_counts=True)
        self.label_ = labels[np.argmax(counts)]
        return self

    def predict(self, X):
        print("model.predict works!")
        return np.full(len(X), self.label_)


# Exclude training rows with blank city labels.
y_train = city_input[:len(X_train), 0]
valid = y_train != ""

pipe = Pipeline([
    ("impute", Imputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", GPT10_5_Astra()),
])

# Fit preprocessing steps and the model.
pipe.fit(X_train[valid], y_train[valid])

# Preprocess test data and predict.
y_pred = pipe.predict(X_test)
print(y_pred[:10])

## 7. Finite-Difference Gradients and Jacobians

Estimate derivatives by slightly changing each input coordinate. Forward differences compare `f(x+h)` with `f(x)`; central differences compare `f(x+h)` with `f(x-h)`.

Use the first standardized numerical row from the CSV pipeline as the evaluation point. This keeps age and income on comparable scales.

In [ ]:
from numcompute.optim import grad, jacobian

x = X_train_ready[0].copy()
print("Evaluation point:", x)

### 7.1 Gradient of a scalar function

For `f(x) = sum(x**2)`, the analytical gradient is `2*x`. The result has the same shape as the one-dimensional input.

In [ ]:
def f(x):
    return np.sum(x ** 2)

central_gradient = grad(f, x, h=1e-5, method="central")
forward_gradient = grad(f, x, h=1e-5, method="forward")
exact_gradient = 2 * x

print("Central gradient:", central_gradient)
print("Forward gradient:", forward_gradient)
print("Analytical gradient:", exact_gradient)
print("Central result matches:", np.allclose(central_gradient, exact_gradient))

### 7.2 Jacobian of a vector function

For `F(x) = [x[0]**2 + x[1], x[0]*x[1]]`, the analytical Jacobian is `[[2*x[0], 1], [x[1], x[0]]]`. Rows correspond to output components and columns to input coordinates.

In [ ]:
def F(x):
    return np.array([
        x[0] ** 2 + x[1],
        x[0] * x[1],
    ])

central_J = jacobian(F, x, method="central")
forward_J = jacobian(F, x, method="forward")
exact_J = np.array([[2 * x[0], 1], [x[1], x[0]]])

print("Central Jacobian:")
print(central_J)
print("Forward Jacobian:")
print(forward_J)
print("Analytical Jacobian:")
print(exact_J)
print("Shape:", central_J.shape)
print("Central result matches:", np.allclose(central_J, exact_J))

## 8. Benchmark (`benchmarking.py`)

TODO (Jose): vectorised vs loop timings; paste the table into README.

In [ ]:
# TODO (Jose)